In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import *
from delta.tables import *

CATALOG = "hive_streamming"

SILVER_SCHEMA = "silver"
GOLD_SCHEMA   = "gold"
STG_SCHEMA    = "gold_stg"

DATASET = "test"

SILVER_TABLE = f"{CATALOG}.{SILVER_SCHEMA}.{DATASET}"

STG_SILVER   = f"{CATALOG}.{STG_SCHEMA}.stg_silver_impacted"

DIM_DATE     = f"{CATALOG}.{GOLD_SCHEMA}.dim_date"
DIM_VIEWER   = f"{CATALOG}.{GOLD_SCHEMA}.dim_viewer"

FACT_EVENT     = f"{CATALOG}.{GOLD_SCHEMA}.fact_qoe_event"
FACT_BUFFERING = f"{CATALOG}.{GOLD_SCHEMA}.fact_buffering_event"
FACT_SESSION   = f"{CATALOG}.{GOLD_SCHEMA}.fact_qoe_session"

WATERMARK_TABLE = f"{CATALOG}.{GOLD_SCHEMA}.gold_watermark"
PIPELINE_NAME = "gold_pipeline"


In [0]:
# SETUP SCHEMAS + WATERMARK

def setup_gold():

    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{GOLD_SCHEMA}")
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{STG_SCHEMA}")

    spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {WATERMARK_TABLE} (
        table_name STRING,
        last_processed_ts TIMESTAMP
    )
    USING DELTA
    """)

    spark.sql(f"""
    INSERT INTO {WATERMARK_TABLE}
    SELECT '{PIPELINE_NAME}', TIMESTAMP('1900-01-01')
    WHERE NOT EXISTS (
        SELECT 1 FROM {WATERMARK_TABLE}
        WHERE table_name = '{PIPELINE_NAME}'
    )
    """)


In [0]:
# READ GOLD WATERMARK

def get_last_gold_ts():
    return (
        spark.table(WATERMARK_TABLE)
        .filter(col("table_name") == PIPELINE_NAME)
        .select("last_processed_ts")
        .collect()[0][0]
    )


In [0]:
# READ SILVER CDF

def read_silver_cdf(last_ts):

    return (
        spark.read.format("delta")
        .option("readChangeFeed", "true")
        .option("startingTimestamp", str(last_ts))
        .table(SILVER_TABLE)
        .filter(col("_change_type").isin("insert", "update_postimage"))
    )


In [0]:
# STAGING — IMPACTED SCOPE ONLY
def build_staging(silver_cdf):

    impacted_keys = (
        silver_cdf
        .select("viewer_id", "filedate")
        .distinct()
    )

    stg = (
        spark.table(SILVER_TABLE)
        .join(impacted_keys, ["viewer_id", "filedate"])
    )

    spark.sql(f"DROP TABLE IF EXISTS {STG_SILVER}")

    stg.write.format("delta").mode("overwrite").saveAsTable(STG_SILVER)


**Dimesional Modelling**

In [0]:
# DIM DATE

def load_dim_date():

    dim = (
        spark.table(STG_SILVER)
        .select("filedate")
        .distinct()
        .withColumn("year", year("filedate"))
        .withColumn("month", month("filedate"))
        .withColumn("day", dayofmonth("filedate"))
        .withColumn("week_of_year", weekofyear("filedate"))
        .withColumn("day_name", date_format("filedate", "EEEE"))
    )

    spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {DIM_DATE} (
        filedate DATE,
        year INT,
        month INT,
        day INT,
        week_of_year INT,
        day_name STRING
    )
    USING DELTA
    """)

    DeltaTable.forName(spark, DIM_DATE) \
        .alias("t") \
        .merge(dim.alias("s"), "t.filedate = s.filedate") \
        .whenNotMatchedInsertAll() \
        .execute()


In [0]:
# DIM VIEWER
def load_dim_viewer():

    dim = (
        spark.table(STG_SILVER)
        .groupBy("viewer_id")
        .agg(
            min("filedate").alias("first_seen_date"),
            max("filedate").alias("last_seen_date")
        )
    )

    spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {DIM_VIEWER} (
        viewer_id STRING,
        first_seen_date DATE,
        last_seen_date DATE
    )
    USING DELTA
    """)

    DeltaTable.forName(spark, DIM_VIEWER) \
        .alias("t") \
        .merge(dim.alias("s"), "t.viewer_id = s.viewer_id") \
        .whenMatchedUpdateAll() \
        .whenNotMatchedInsertAll() \
        .execute()


_**Fact**_ **_tables_**

In [0]:
# FACT — EVENT (TRANSACTIONAL / AD-HOC)
def load_fact_event():

    df = spark.table(STG_SILVER)

    spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {FACT_EVENT} (
        event_id STRING,
        viewer_id STRING,
        filedate DATE,
        snapshot_index INT,
        bitrate_mbps DOUBLE,
        quality_switch_flag INT,
        is_buffering BOOLEAN,
        buffer_duration_sec DOUBLE
    )
    USING DELTA
    """)

    DeltaTable.forName(spark, FACT_EVENT) \
        .alias("t") \
        .merge(df.alias("s"), "t.event_id = s.event_id") \
        .whenMatchedUpdateAll() \
        .whenNotMatchedInsertAll() \
        .execute()


In [0]:
# FACT — BUFFERING (POWER BI)

def load_fact_buffering():

    s = spark.table(STG_SILVER)

    w = Window.partitionBy("viewer_id", "filedate").orderBy("snapshot_index")

    s = (
        s.withColumn("is_start", col("start_buffering_ts").isNotNull().cast("int"))
         .withColumn("buffer_session_id", sum("is_start").over(w))
    )

    fact = (
        s.groupBy("viewer_id", "filedate", "buffer_session_id")
        .agg(
            min("start_buffering_ts").alias("buffer_start_ts"),
            max("end_buffering_ts").alias("buffer_end_ts")
        )
        .filter(col("buffer_start_ts").isNotNull() & col("buffer_end_ts").isNotNull())
        .withColumn(
            "buffer_duration_sec",
            col("buffer_end_ts").cast("long") - col("buffer_start_ts").cast("long")
        )
    )

    spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {FACT_BUFFERING} (
        viewer_id STRING,
        filedate DATE,
        buffer_session_id INT,
        buffer_start_ts TIMESTAMP,
        buffer_end_ts TIMESTAMP,
        buffer_duration_sec DOUBLE
    )
    USING DELTA
    """)

    DeltaTable.forName(spark, FACT_BUFFERING) \
        .alias("t") \
        .merge(
            fact.alias("s"),
            "t.viewer_id = s.viewer_id AND "
            "t.filedate = s.filedate AND "
            "t.buffer_session_id = s.buffer_session_id"
        ) \
        .whenMatchedUpdateAll() \
        .whenNotMatchedInsertAll() \
        .execute()


In [0]:
from pyspark.sql.functions import count, sum, avg, min, max, col

def load_fact_session():

    base = (
        spark.table(STG_SILVER)
        .groupBy("viewer_id", "filedate")
        .agg(
            count("*").alias("total_events"),
            sum("quality_switch_flag").alias("quality_switch_count"),
            avg("bitrate_mbps").alias("avg_bitrate_mbps"),
            (min("bitrate_mbps") * 1000).alias("min_bitrate_kbps"),
            (max("bitrate_mbps") * 1000).alias("max_bitrate_kbps")
        )
    )

    buffering = (
        spark.table(FACT_BUFFERING)
        .groupBy("viewer_id", "filedate")
        .agg(
            count("*").alias("buffering_event_count"),
            sum("buffer_duration_sec").alias("total_buffering_sec")
        )
    )

    fact = (
        base.join(buffering, ["viewer_id", "filedate"], "left")
        .fillna(0)
    )

    spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {FACT_SESSION} (
        viewer_id STRING,
        filedate DATE,
        total_events INT,
        quality_switch_count INT,
        avg_bitrate_mbps DOUBLE,
        min_bitrate_kbps DOUBLE,
        max_bitrate_kbps DOUBLE,
        buffering_event_count INT,
        total_buffering_sec DOUBLE
    )
    USING DELTA
    """)

    DeltaTable.forName(spark, FACT_SESSION) \
        .alias("t") \
        .merge(
            fact.alias("s"),
            "t.viewer_id = s.viewer_id AND t.filedate = s.filedate"
        ) \
        .whenMatchedUpdateAll() \
        .whenNotMatchedInsertAll() \
        .execute()


In [0]:

# UPDATE WATERMARK

def update_watermark(silver_cdf):

    new_ts = silver_cdf.select(max("_commit_timestamp")).collect()[0][0]

    spark.sql(f"""
    UPDATE {WATERMARK_TABLE}
    SET last_processed_ts = TIMESTAMP('{new_ts}')
    WHERE table_name = '{PIPELINE_NAME}'
    """)



In [0]:
def main():

    setup_gold()

    last_ts = get_last_gold_ts()

    silver_cdf = read_silver_cdf(last_ts)

    if silver_cdf.limit(1).count() == 0:
        print("No Silver changes → Gold skipped")
        return

    # 1️⃣ STAGING
    build_staging(silver_cdf)

    # 2️⃣ DIMENSIONS (FIRST)
    load_dim_date()
    load_dim_viewer()

    # 3️⃣ FACTS (AFTER DIMS)
    load_fact_event()
    load_fact_buffering()
    load_fact_session()

    # 4️⃣ WATERMARK
    update_watermark(silver_cdf)

    print("✅ GOLD PIPELINE COMPLETED WITH REFERENTIAL INTEGRITY")


In [0]:
main()